# Nismara Smart Analytics: Laporan Perbandingan 360° (Month vs Month)
Versi diperluas 360°: Membandingkan dua bulan penuh secara terpisah antara **Divisi Logistik (Truck)** vs **Divisi Coaches (Bus)**, plus seluruh ekosistem komunitas (`convoylobby`, `contracts`, `ncevents`, `coupons`, `giveaways`, `giveawaytickets`, `uservouchers`, `pointhistories`), **Minigames/NC Sink** (`lottoperiods`, `lottotickets`, `scratchtickets`, `racingtickets`, `quizattempts`), **Ekonomi & BBM** (`transactions`, `fueltransactions`), dan **Galeri** (`gallery_posts`).

In [ ]:
!pip install mcp python-dotenv pandas google-generativeai

In [ ]:
import asyncio
import os
import json
import pandas as pd
import google.generativeai as genai
from dotenv import load_dotenv
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel('gemini-3.5-flash')

server_params = StdioServerParameters(
    command="node",
    args=["build/index.js"],
    env={
        **os.environ,
        "MONGODB_URI": os.getenv("MONGODB_URI"),
        "MCP_API_TOKEN": os.getenv("MCP_API_TOKEN")
    }
)

MONTH_TO_NUM = {
    "januari": 1, "februari": 2, "maret": 3, "april": 4,
    "mei": 5, "juni": 6, "juli": 7, "agustus": 8,
    "september": 9, "oktober": 10, "november": 11, "desember": 12,
    "january": 1, "february": 2, "march": 3, "may": 5,
    "june": 6, "july": 7, "august": 8, "october": 10, "december": 12,
}

COLLECTION_DATE_FIELDS = [
    ("jobhistories", "createdAt"),
    ("convoylobby", "startDate"),
    ("contracts", "startDate"),
    ("ncevents", "createdAt"),
    ("coupons", "startDate"),
    ("giveaways", "startDate"),
    ("giveawaytickets", "createdAt"),
    ("uservouchers", "createdAt"),
    ("pointhistories", "createdAt"),
    ("lottoperiods", "startDate"),
    ("lottotickets", "createdAt"),
    ("scratchtickets", "createdAt"),
    ("racingtickets", "createdAt"),
    ("quizattempts", "createdAt"),
    ("transactions", "createdAt"),
    ("fueltransactions", "createdAt"),
    ("gallery_posts", "createdAt"),
]

def get_month_folder_key(bulan, tahun):
    m = int(bulan) if str(bulan).isdigit() else MONTH_TO_NUM.get(str(bulan).strip().lower(), 0)
    return f"{tahun}-{m:02d}" if m else f"{tahun}-{str(bulan).strip().lower()}"

def get_month_report_dir(bulan, tahun):
    folder = os.path.join("report", get_month_folder_key(bulan, tahun))
    os.makedirs(folder, exist_ok=True)
    return folder

def resolve_month_file(bulan, tahun, filename):
    primary = os.path.join(get_month_report_dir(bulan, tahun), filename)
    if os.path.exists(primary):
        return primary
    legacy_report = os.path.join("report", filename)
    if os.path.exists(legacy_report):
        return legacy_report
    return filename


### TAHAP 1: Ekspor Multi-Koleksi 360° (2 Bulan)
Menarik 17 koleksi aktif untuk DUA bulan sekaligus dan menyimpannya rapi ke `report/YYYY-MM/`.

In [ ]:
async def fetch_collection(session, collection_name, query_json):
    result = await session.call_tool("query_collection", arguments={
        "collectionName": collection_name,
        "query": query_json,
        "limit": 10000,
        "sort": '{"_id": -1}'
    })
    data_mentah = result.content[0].text
    return [] if data_mentah == "[]" else json.loads(data_mentah)

async def export_all_data(bulan, tahun, gte_date, lte_date):
    async with stdio_client(server_params, errlog=None) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            report_dir = get_month_report_dir(bulan, tahun)
            print(f"\n🔍 Menarik data 360° {bulan} {tahun} -> {report_dir}/ ...")
            
            for col, date_field in COLLECTION_DATE_FIELDS:
                query_json = f'{{ "{date_field}": {{ "$gte": "{gte_date}", "$lte": "{lte_date}" }} }}'
                data = await fetch_collection(session, col, query_json)
                filename = os.path.join(report_dir, f"data_{col}_{bulan.lower()}{tahun}.json")
                
                if col == "jobhistories":
                    if not data:
                        print(f" ├── [jobhistories] 0 data ditemukan.")
                        continue
                    for item in data:
                        stops = item.get("stops")
                        item["stopsCount"] = len(stops) if isinstance(stops, list) else 0
                        if not item.get("jobCategory"):
                            mt = str(item.get("marketType") or "").lower()
                            item["jobCategory"] = "coach" if mt.startswith("bus") else "truck"

                    df = pd.json_normalize(data)
                    kolom_penting = [
                        'jobId', 'driverId', 'truckyId', 'jobStatus', 'jobCategory',
                        'game', 'gameMode', 'isHardcore', 'hardcoreRating', 'marketType',
                        'createdAt', 'completedAt', 'cargoName', 'cargoMass',
                        'totalPassengers', 'stopsCount', 'vehicleStatus',
                        'sourceCity', 'destinationCity', 'distanceKm', 'durationSeconds',
                        'nc.total', 'nc.base', 'nc.special', 'nc.hardcore', 'nc.event',
                        'nc.userBoost', 'nc.booster', 'nc.nismaraplus', 'nc.truckersmp', 'nc.haulmp',
                        'xp.total', 'discount.total', 'tax.amount',
                        'damage.vehicle', 'damage.trailer', 'damage.cargo',
                        'penalty.vehicle', 'penalty.trailer', 'penalty.cargo',
                        'penalty.speed', 'penalty.distance', 'penalty.total',
                        'ncCost.rent', 'ncCost.service', 'ncCost.fuel', 'ncCost.fines', 'ncCost.total',
                        'vehicle.brand', 'vehicle.model'
                    ]
                    kolom_tersedia = [c for c in kolom_penting if c in df.columns]
                    df[kolom_tersedia].to_csv(filename.replace(".json", ".csv"), index=False, encoding="utf-8-sig")
                    print(f" ├── [{col}] {len(data)} data diekspor ke CSV bersih.")
                else:
                    with open(filename, "w", encoding="utf-8") as f:
                        json.dump(data, f, ensure_ascii=False, indent=2)
                    print(f" ├── [{col}] {len(data)} data diekspor ke JSON.")

# ==================================================
# SESUAIKAN DUA BULAN YANG INGIN DIBANDINGKAN
# ==================================================
BULAN_A, TAHUN_A, GTE_A, LTE_A = "Oktober", "2026", "2026-10-01T00:00:00Z", "2026-10-31T23:59:59Z"
BULAN_B, TAHUN_B, GTE_B, LTE_B = "November", "2026", "2026-11-01T00:00:00Z", "2026-11-30T23:59:59Z"

async def step1_export():
    await export_all_data(BULAN_A, TAHUN_A, GTE_A, LTE_A)
    await export_all_data(BULAN_B, TAHUN_B, GTE_B, LTE_B)

await step1_export()

### TAHAP 2: Agregasi Data Ekosistem 360° & Laporan Perbandingan Gemini
Membaca seluruh file dari kedua bulan dan menyusun Laporan Perbandingan Komprehensif ke `report/comparisons/`.

In [ ]:
def baca_json(filepath):
    if not os.path.exists(filepath):
        return []
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
        return data if isinstance(data, list) else []

def _classify_category(df):
    if 'jobCategory' in df.columns:
        cat = df['jobCategory'].fillna('').astype(str).str.lower()
    else:
        cat = pd.Series([''] * len(df), index=df.index)
    if 'marketType' in df.columns:
        mt = df['marketType'].fillna('').astype(str).str.lower()
        is_coach = (cat == 'coach') | mt.str.startswith('bus')
    else:
        is_coach = (cat == 'coach')
    return df[~is_coach].copy(), df[is_coach].copy()

def _summarize_division(df_div, div_type="truck"):
    if df_div.empty:
        return "Belum ada data pengiriman pada divisi ini di bulan tersebut."

    total_all = len(df_div)
    df_comp = df_div[df_div['jobStatus'] == 'COMPLETED'] if 'jobStatus' in df_div.columns else df_div
    df_canc = df_div[df_div['jobStatus'] == 'CANCELED'] if 'jobStatus' in df_div.columns else df_div.iloc[0:0]

    total_jobs = len(df_comp)
    canceled_jobs = len(df_canc)
    comp_rate = (total_jobs / total_all * 100) if total_all > 0 else 0

    total_distance = df_comp['distanceKm'].sum() if 'distanceKm' in df_comp.columns else 0
    total_revenue = df_comp['nc.total'].sum() if 'nc.total' in df_comp.columns else 0
    total_xp = df_comp['xp.total'].sum() if 'xp.total' in df_comp.columns else 0
    total_cost = df_div['ncCost.total'].sum() if 'ncCost.total' in df_div.columns else 0
    total_discount = df_comp['discount.total'].sum() if 'discount.total' in df_comp.columns else 0
    total_penalty = df_div['penalty.total'].sum() if 'penalty.total' in df_div.columns else 0
    total_hours = (df_comp['durationSeconds'].sum() / 3600) if 'durationSeconds' in df_comp.columns else 0
    unique_drivers = df_comp['driverId'].nunique() if 'driverId' in df_comp.columns else 0

    game_lines = []
    if 'game' in df_div.columns and 'jobStatus' in df_div.columns and 'nc.total' in df_div.columns:
        for g in df_div['game'].dropna().unique():
            sub = df_div[df_div['game'] == g]
            c_ok = (sub['jobStatus'] == 'COMPLETED').sum()
            c_no = (sub['jobStatus'] == 'CANCELED').sum()
            rev = sub[sub['jobStatus'] == 'COMPLETED']['nc.total'].sum()
            game_lines.append(f"{g}: {c_ok} Sukses, {c_no} Batal | Pendapatan: {rev:,.2f} NC")
    game_text = " | ".join(game_lines) if game_lines else "-"

    mode_text = "-"
    if 'gameMode' in df_div.columns:
        mc = df_div['gameMode'].fillna('unknown').value_counts()
        mode_text = ", ".join([f"{k}: {v} jobs" for k, v in mc.items()])

    market_text = "-"
    if 'marketType' in df_div.columns:
        mk = df_div['marketType'].fillna('unknown').value_counts()
        market_text = ", ".join([f"{k}: {v} jobs" for k, v in mk.items()])

    veh_text = "-"
    if 'vehicleStatus' in df_div.columns:
        vs = df_div['vehicleStatus'].fillna('unknown').value_counts()
        veh_text = ", ".join([f"{k}: {v} jobs" for k, v in vs.items()])

    avg_dmg_v = df_comp['damage.vehicle'].mean() if 'damage.vehicle' in df_comp.columns and len(df_comp) else 0
    avg_dmg_t = df_comp['damage.trailer'].mean() if 'damage.trailer' in df_comp.columns and len(df_comp) else 0
    avg_dmg_c = df_comp['damage.cargo'].mean() if 'damage.cargo' in df_comp.columns and len(df_comp) else 0
    hc_jobs = int((df_comp['isHardcore'] == True).sum()) if 'isHardcore' in df_comp.columns else 0

    extra_lines = []
    if div_type == "coach":
        total_pax = int(df_comp['totalPassengers'].sum()) if 'totalPassengers' in df_comp.columns else 0
        avg_pax = (total_pax / total_jobs) if total_jobs > 0 else 0
        total_stops = int(df_comp['stopsCount'].sum()) if 'stopsCount' in df_comp.columns else 0
        top_buses = "-"
        if 'vehicle.brand' in df_comp.columns and 'vehicle.model' in df_comp.columns:
            buses = (df_comp['vehicle.brand'].fillna('') + ' ' + df_comp['vehicle.model'].fillna('')).str.strip()
            top_b = buses[buses != ''].value_counts().head(5)
            if len(top_b):
                top_buses = ", ".join([f"{k} ({v} trip)" for k, v in top_b.items()])
        top_routes = "-"
        if 'sourceCity' in df_comp.columns and 'destinationCity' in df_comp.columns:
            routes = (df_comp['sourceCity'].fillna('?') + ' -> ' + df_comp['destinationCity'].fillna('?'))
            top_r = routes.value_counts().head(5)
            if len(top_r):
                top_routes = ", ".join([f"{k} ({v} trip)" for k, v in top_r.items()])
        extra_lines.append(f"    - Total Penumpang Diangkut: {total_pax:,} penumpang (Rata-rata {avg_pax:.1f} pax/trip)")
        extra_lines.append(f"    - Total Pemberhentian/Halte (Stops): {total_stops:,} stops")
        extra_lines.append(f"    - Top Armada Bus: {top_buses}")
        extra_lines.append(f"    - Top Rute Bus: {top_routes}")
    else:
        top_cargo = "-"
        if 'cargoName' in df_comp.columns and 'nc.total' in df_comp.columns:
            top_5 = df_comp.groupby('cargoName')['nc.total'].sum().sort_values(ascending=False).head(5)
            if len(top_5):
                top_cargo = ", ".join([f"{k} ({v:,.0f} NC)" for k, v in top_5.items()])
        extra_lines.append(f"    - Top 5 Kargo (Pendapatan NC): {top_cargo}")
        extra_lines.append(f"    - Mode Hardcore: {hc_jobs:,} pengiriman sukses")

    driver_line = f" | Driver Aktif Unik: {unique_drivers}" if unique_drivers > 0 else ""
    extra_block = "\n".join(extra_lines)

    return f"""    - Total Pengiriman: {total_jobs:,} Sukses, {canceled_jobs:,} Batal (Completion Rate: {comp_rate:.1f}%){driver_line}
    - Total Jarak & Durasi: {total_distance:,.2f} Km | {total_hours:,.1f} Jam
    - Total Pendapatan Kotor: {total_revenue:,.2f} NC | XP: {total_xp:,.0f} XP
    - Biaya Operasional: {total_cost:,.2f} NC | Penghematan Diskon: {total_discount:,.2f} NC
    - Rata-rata Kerusakan: Kendaraan {avg_dmg_v:.2f}%, Trailer {avg_dmg_t:.2f}%, Kargo/Penumpang {avg_dmg_c:.2f}% | Penalti Sistem: {total_penalty:,.1f} Poin
    - Rincian Per Game: {game_text}
    - Platform Bermain: {mode_text}
    - Tipe Pasar: {market_text}
    - Kepemilikan Armada: {veh_text}
{extra_block}"""

def hitung_statistik_terpisah(bulan, tahun):
    b_t = f"{bulan.lower()}{tahun}"
    csv_job = resolve_month_file(bulan, tahun, f"data_jobhistories_{b_t}.csv")
    if not os.path.exists(csv_job):
        return f"Data {bulan} {tahun} tidak ditemukan. Jalankan Tahap 1 terlebih dahulu."

    df_job = pd.read_csv(csv_job)
    df_truck, df_coach = _classify_category(df_job)
    truck_summary = _summarize_division(df_truck, "truck")
    coach_summary = _summarize_division(df_coach, "coach")

    convoys = baca_json(resolve_month_file(bulan, tahun, f"data_convoylobby_{b_t}.json"))
    contracts = baca_json(resolve_month_file(bulan, tahun, f"data_contracts_{b_t}.json"))
    if not contracts:
        contracts = baca_json(resolve_month_file(bulan, tahun, f"data_contracthistories_{b_t}.json"))
    ncevents = baca_json(resolve_month_file(bulan, tahun, f"data_ncevents_{b_t}.json"))
    coupons = baca_json(resolve_month_file(bulan, tahun, f"data_coupons_{b_t}.json"))
    giveaways = baca_json(resolve_month_file(bulan, tahun, f"data_giveaways_{b_t}.json"))
    giveaway_tickets = baca_json(resolve_month_file(bulan, tahun, f"data_giveawaytickets_{b_t}.json"))
    vouchers = baca_json(resolve_month_file(bulan, tahun, f"data_uservouchers_{b_t}.json"))
    pointhistories = baca_json(resolve_month_file(bulan, tahun, f"data_pointhistories_{b_t}.json"))

    convoy_participants = sum([len(c.get('partisipan', [])) for c in convoys])
    total_contract_jobs = sum([c.get('completedContracts', 0) for c in contracts])
    total_contract_nc = sum([c.get('totalNCEarned', 0) for c in contracts])
    nc_event_participants = sum([len(e.get('participants', [])) for e in ncevents])
    total_coupon_claims = sum([len(c.get('driverClaims', [])) for c in coupons])
    total_coupon_nc = sum([c.get('totalNcClaimed', 0) for c in coupons])

    gw_winners = sum([len(g.get('winners', [])) for g in giveaways])
    gw_nc_burned = sum([(g.get('stats') or {}).get('totalNcBurned', 0) for g in giveaways])
    gw_ticket_nc = sum([t.get('costNC', 0) for t in giveaway_tickets])
    gw_titles = ", ".join([g.get('title', '-') for g in giveaways]) if giveaways else "-"

    vouchers_used = sum([1 for v in vouchers if str(v.get('status', '')).upper() == 'USED'])
    penalty_added = sum([p.get('points', 0) for p in pointhistories if p.get('type') == 'add'])
    penalty_paid = sum([p.get('points', 0) for p in pointhistories if p.get('type') == 'remove'])

    lotto_periods = baca_json(resolve_month_file(bulan, tahun, f"data_lottoperiods_{b_t}.json"))
    lotto_tickets = baca_json(resolve_month_file(bulan, tahun, f"data_lottotickets_{b_t}.json"))
    scratch_tickets = baca_json(resolve_month_file(bulan, tahun, f"data_scratchtickets_{b_t}.json"))
    racing_tickets = baca_json(resolve_month_file(bulan, tahun, f"data_racingtickets_{b_t}.json"))
    quiz_attempts = baca_json(resolve_month_file(bulan, tahun, f"data_quizattempts_{b_t}.json"))

    lotto_prize = sum([t.get('prizeWon', 0) for t in lotto_tickets])
    scratch_spent = sum([t.get('price', 0) for t in scratch_tickets])
    scratch_won = sum([t.get('prizeWon', 0) for t in scratch_tickets])
    racing_bet = sum([t.get('betAmount', 0) for t in racing_tickets])
    racing_won = sum([t.get('prizeWon', 0) for t in racing_tickets])
    quiz_passed = sum([1 for q in quiz_attempts if q.get('passed') == True])
    quiz_pass_rate = (quiz_passed / len(quiz_attempts) * 100) if quiz_attempts else 0

    transactions = baca_json(resolve_month_file(bulan, tahun, f"data_transactions_{b_t}.json"))
    fuel_tx = baca_json(resolve_month_file(bulan, tahun, f"data_fueltransactions_{b_t}.json"))
    gallery_posts = baca_json(resolve_month_file(bulan, tahun, f"data_gallery_posts_{b_t}.json"))

    tx_success = [t for t in transactions if str(t.get('status', 'success')).lower() in ('success', 'completed', 'paid')]
    tx_nc = sum([t.get('amount', 0) for t in tx_success if str(t.get('currency', 'NC')).upper() == 'NC'])
    tx_idr = sum([t.get('amount', 0) for t in tx_success if str(t.get('currency', '')).upper() == 'IDR'])
    tx_voucher_disc = sum([(t.get('metadata') or {}).get('voucherDiscount', 0) for t in tx_success])

    fuel_liters = int(sum([f.get('amount', 0) for f in fuel_tx]))
    fuel_p2p = int(sum([f.get('amount', 0) for f in fuel_tx if f.get('type') == 'p2p']))
    fuel_sys = int(sum([f.get('amount', 0) for f in fuel_tx if f.get('type') == 'system']))
    fuel_vol_nc = sum([f.get('totalPrice', 0) for f in fuel_tx])
    fuel_fee_nc = sum([f.get('fee', 0) for f in fuel_tx])
    gallery_likes = sum([len(p.get('likes', [])) for p in gallery_posts])

    return f"""
STATISTIK 360° {bulan.upper()} {tahun} (TERPISAH: LOGISTIK TRUCK vs COACHES BUS):

[1. DIVISI LOGISTIK — TRUCK / KARGO]
{truck_summary}

[2. DIVISI COACHES — BUS / PENUMPANG (DLC COACHES)]
{coach_summary}

[3. EKOSISTEM KOMUNITAS, EVENT & PROMO]
    - Convoy Diadakan: {len(convoys)} Convoy (Total {convoy_participants:,} partisipan)
    - Special Contracts: {len(contracts)} kontrak | {total_contract_jobs:,} pengiriman senilai {total_contract_nc:,.2f} NC
    - NC Events (Boost): {len(ncevents)} event (Total {nc_event_participants:,} peserta)
    - Giveaway: {len(giveaways)} event ({gw_titles}) | {len(giveaway_tickets):,} tiket | {gw_winners} pemenang | NC terbakar: {max(gw_nc_burned, gw_ticket_nc):,.2f} NC
    - Voucher User: {len(vouchers):,} diterbitkan ({vouchers_used:,} terpakai)
    - Penebaran Kupon: {len(coupons)} dirilis, {total_coupon_claims:,} diklaim ({total_coupon_nc:,.2f} NC)
    - Kedisiplinan Driver: {penalty_paid:,} poin penalti dilunasi vs {penalty_added:,} poin penalti baru masuk (sistem)

[4. MINIGAMES & PERPUTARAN NC (NC SINK)]
    - Lotto: {len(lotto_periods)} periode | {len(lotto_tickets):,} tiket | Hadiah: {lotto_prize:,.2f} NC
    - Scratchers: {len(scratch_tickets):,} tiket | Belanja: {scratch_spent:,.2f} NC vs Hadiah: {scratch_won:,.2f} NC (Net Burn: {scratch_spent - scratch_won:,.2f} NC)
    - Truck Racing: {len(racing_tickets):,} taruhan | Taruhan: {racing_bet:,.2f} NC vs Hadiah: {racing_won:,.2f} NC (Net Burn: {racing_bet - racing_won:,.2f} NC)
    - Kuis Driver: {len(quiz_attempts):,} percobaan | {quiz_passed:,} lulus ({quiz_pass_rate:.1f}%)

[5. EKONOMI BELANJA, PASAR BBM & GALERI SOSIAL]
    - Transaksi Toko/Fleet/Market: {len(tx_success):,} transaksi | Belanja: {tx_nc:,.2f} NC (+ Rp {tx_idr:,.0f} IDR) | Diskon Voucher: {tx_voucher_disc:,.2f} NC
    - Pasar BBM: {len(fuel_tx):,} transaksi | {fuel_liters:,} L (Sistem: {fuel_sys:,} L, P2P: {fuel_p2p:,} L) | Volume: {fuel_vol_nc:,.2f} NC | Fee: {fuel_fee_nc:,.2f} NC
    - Galeri Komunitas: {len(gallery_posts):,} postingan ({gallery_likes:,} likes)
"""

def buat_laporan_komprehensif():
    summary_a = hitung_statistik_terpisah(BULAN_A, TAHUN_A)
    summary_b = hitung_statistik_terpisah(BULAN_B, TAHUN_B)
    
    print(f"📊 Seluruh data 360° ({BULAN_A} {TAHUN_A} vs {BULAN_B} {TAHUN_B}) berhasil dirangkum. Menyerahkan ke Gemini...")
    
    prompt = f"""
    Anda adalah Senior Data Analyst & Community Manager di Nismara Transport.
    Berikut adalah rangkuman data operasional murni dan ekosistem 360° kami untuk DUA BULAN:
    
    {summary_a}
    
    {summary_b}
    
    TUGAS ANDA:
    Buatlah "Laporan Aktivitas Komprehensif Perusahaan ({BULAN_A} {TAHUN_A} vs {BULAN_B} {TAHUN_B})" untuk Direktur Utama.
    Laporan harus tajam, profesional, menggunakan bullet-points, dan mencakup:
    1. Perbandingan Divisi Logistik (Truck) vs Divisi Coaches (Bus) secara TERPISAH.
    2. Evaluasi Efisiensi, Mode Hardcore & Keselamatan (SP 10% vs TMP 20% damage tolerance, penalti sistem, diskon asuransi).
    3. Ekosistem Komunitas, Event, Giveaway & Voucher (Convoy, Special Contracts, NC Boost, Giveaway & NC Burn, Voucher, Kupon, Penalti).
    4. Kesehatan Ekonomi Internal, Minigames (Lotto/Scratchers/Racing/Kuis), Pasar BBM (Sistem vs P2P) & Galeri.
    5. Kesimpulan & 3 Rekomendasi Strategis konkret ke depan.
    
    PENTING:
    - PISAHKAN secara tegas analisa Divisi Logistik (Truck/Kargo) dan Divisi Coaches (Bus/Penumpang). JANGAN mencampurkan statistik penumpang/bus dengan kargo truk.
    - Semua nilai uang virtual menggunakan "NC" (Nismara Coin), bukan USD ($). Uang nyata pakai "IDR / Rp".
    - Angka damage adalah PERSENTASE KERUSAKAN, bukan Poin.
    - Bandingkan angka secara eksplisit (Misal: "Pendapatan naik dari X menjadi Y").
    """
    
    response = model.generate_content(prompt)
    print("\n=======================================================")
    print("LAPORAN ANALISA KOMPREHENSIF 360° GEMINI (PERBANDINGAN)")
    print("=======================================================\n")
    print(response.text)
    comp_dir = os.path.join("report", "comparisons")
    os.makedirs(comp_dir, exist_ok=True)
    slug = f"{BULAN_A.lower()}{TAHUN_A}_vs_{BULAN_B.lower()}{TAHUN_B}"
    md_out = os.path.join(comp_dir, f"laporan_perbandingan_{slug}.md")
    with open(md_out, "w", encoding="utf-8") as f:
        f.write(response.text)
    print(f"\n💾 Laporan perbandingan otomatis disimpan ke: {md_out}")

buat_laporan_komprehensif()
